# Clinical Sample Selection and Compile L3 AIFI cell types  

Subset L3 AIFI cell type objects based on 141 selected clinical samples

Concatenate L3 cell type objects together to compiled anndata object

## Load libraries

In [1]:
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=RuntimeWarning)

import concurrent.futures
from concurrent.futures import ThreadPoolExecutor
from datetime import date
import hisepy
import os
import pandas as pd
import re
import scanpy as sc
import scanpy.external as sce
import anndata as ad
import glob
import random


In [2]:
out_dir = 'output_sample_selection'
if not os.path.isdir(out_dir):
    os.makedirs(out_dir)

## Helper functions

These make it a bit simpler to cache and read in files from HISE

In [3]:
def cache_uuid_path(uuid):
    cache_path = '/home/jupyter/cache/{u}'.format(u = uuid)
    if not os.path.isdir(cache_path):
        hise_res = hisepy.reader.cache_files([uuid])
    filename = os.listdir(cache_path)[0]
    cache_file = '{p}/{f}'.format(p = cache_path, f = filename)
    return cache_file

In [4]:
def read_parquet_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = pd.read_parquet(cache_file)
    return res

In [5]:
def sort_adata_uuid(uuid, sort_cols = ['AIFI_L2', 'sample.sampleKitGuid']):
    cache_file = cache_uuid_path(uuid)
    adata = sc.read_h5ad(cache_file)
    obs = adata.obs
    obs = obs.sort_values(sort_cols)
    adata = adata[obs.index]
    adata.write_h5ad(cache_file)

This function will enable us to connect to our .h5ad files without loading the entire thing into memory. We'll then load only the cells that we want for each cell class to assemble them for writing. This should save us some overhead as we do our subsetting.

In [6]:
def read_adata_backed_uuid(uuid):
    cache_file = cache_uuid_path(uuid)
    res = sc.read_h5ad(cache_file, backed = 'r')
    return res

This function will remove Ig-related genes, which is recommended by Marla Glass for analysis of B cell subtypes

In [7]:
def remove_ig_genes(adata):
    igl_genes = [gene for gene in adata.var_names if gene.startswith("IGL")]
    igk_genes = [gene for gene in adata.var_names if gene.startswith("IGK")]
    ighc_genes = [gene for gene in adata.var_names if gene.startswith("IGH")]
    exl_genes = igl_genes + igk_genes + ighc_genes

    filtered_genes = [gene for gene in adata.var_names if gene not in exl_genes]
    adata = adata[:, filtered_genes]

    return adata

This function will apply a standard normalization, nearest neighbors, clustering, and UMAP process to our cell subsets:

In [8]:
def process_adata(adata, resolution = 2):
    
    # Keep a copy of the raw data
    adata = adata.raw.to_adata()
    adata.raw = adata

    print('Normalizing', end = "; ")
    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)

    print('Finding HVGs', end = "; ")
    # Restrict downstream steps to variable genes
    sc.pp.highly_variable_genes(adata)
    adata = adata[:, adata.var_names[adata.var['highly_variable']]].copy()

    print('Scaling', end = "; ")
    # Scale variable genes
    sc.pp.scale(adata)

    print('PCA', end = "; ")
    # Run PCA
    sc.tl.pca(adata, svd_solver = 'arpack')
    
    print('Neighbors', end = "; ")
    # Find nearest neighbors
    sc.pp.neighbors(
        adata, 
        n_neighbors = 50,
        n_pcs = 30
    )

    print('Leiden', end = "; ")
    # Find clusters
    sc.tl.leiden(
        adata, 
        resolution = resolution, 
        key_added = 'leiden_{r}'.format(r = resolution),
        n_iterations = 2
    )

    print('UMAP', end = "; ")
    # Run UMAP
    sc.tl.umap(adata, min_dist = 0.05)
    
    print('Renormalizing')
    adata = adata.raw.to_adata()
    adata.raw = adata

    # Normalize and log transform
    sc.pp.normalize_total(adata)
    sc.pp.log1p(adata)
    
    return adata

In [9]:
def format_cell_type(cell_type):
    cell_type = re.sub('\\+', 'pos', cell_type)
    cell_type = re.sub('-', 'neg', cell_type)
    cell_type = re.sub(' ', '_', cell_type)
    return cell_type

In [10]:
def element_id(n = 3):
    import periodictable
    from random import randrange
    rand_el = []
    for i in range(n):
        el = randrange(0,118)
        rand_el.append(periodictable.elements[el].name)
    rand_str = '-'.join(rand_el)
    return rand_str

In [11]:
# make a function to find files
def get_filepaths_with_glob(root_path: str, file_regex: str):
    return glob.glob(os.path.join(root_path, file_regex))

In [12]:
# Define a function to extract the desired substring using regex
def extract_substring(path, pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'):
    #pattern = r'preRA_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad'
    match = re.search(pattern, path)
    if match:
        return match.group(1)
    return None

In [13]:
def read_anndata_files(file_tuples):
    """
    Read Anndata objects from H5AD files and store them in a dictionary with custom names.

    Parameters:
        file_tuples (list of tuples): List of tuples where each tuple contains filename and desired name.

    Returns:
        dict: Dictionary containing Anndata objects with custom names.
    """
    anndata_dict = {}
    for filename, name in file_tuples:
        anndata_obj = anndata.read_h5ad(filename)
        anndata_dict[name] = anndata_obj
    return anndata_dict

In [14]:
def reformat_cell_type(cell_type):
    '''convert cell type names read in from file back to original L3 labels'''
    cell_type = re.sub('pos', '+', cell_type)
    cell_type = re.sub('neg', '-', cell_type)
    cell_type = re.sub('_',' ', cell_type)
    return cell_type

## Identify files for use in HISE

In [17]:
search_id = 'mercury-nickel-mendelevium'

Retrieve files stored in our HISE project store

In [19]:
ps_df = hisepy.list_files_in_project_store('Dyna_IHandA')
ps_df = ps_df[['id', 'name']]

Filter for files from the previous notebook using our search_id

In [20]:
search_df = ps_df[ps_df['name'].str.contains(search_id)]
search_df = search_df.sort_values('name')

In [21]:
h5ad_df = search_df[search_df['name'].str.contains('.h5ad')]

In [22]:
h5ad_df

,id,name
75,accae6f5-2277-4237-9283-7aac54e42123,mercury-nickel-mendelevium/up1_cluster_harmoni...
76,9db395cc-811e-4aa6-a502-6da51700f4d0,mercury-nickel-mendelevium/up1_cluster_harmoni...
77,16fbf925-061d-48d2-8539-f4063dfe6791,mercury-nickel-mendelevium/up1_cluster_harmoni...
78,17cddbe8-1768-454d-a5c1-47344cd92863,mercury-nickel-mendelevium/up1_cluster_harmoni...
79,785e01a1-2fbd-42b3-a560-0dfe91a32f1c,mercury-nickel-mendelevium/up1_cluster_harmoni...
...,...,...
110,50b3c708-2cc8-4e43-989e-d5b167ad0eff,mercury-nickel-mendelevium/up1_cluster_harmoni...
111,2e8c875d-dbbf-4447-852c-8872f44c9a11,mercury-nickel-mendelevium/up1_cluster_harmoni...
112,f191b7a2-e072-4059-98a2-a60479250de3,mercury-nickel-mendelevium/up1_cluster_harmoni...
113,084f6e50-14d3-44e6-a524-05d47ff25808,mercury-nickel-mendelevium/up1_cluster_harmoni...


In [23]:
h5ad_uuids = {}
for i in range(h5ad_df.shape[0]):
    fn = h5ad_df['name'].tolist()[i]
    group_name = re.sub('.+pbmc_', '', fn)
    group_name = re.sub('_init.+', '', group_name)
    h5ad_uuids[group_name] = h5ad_df['id'].tolist()[i]

In [24]:
h5ad_uuids

{'mercury-nickel-mendelevium/up1_cluster_harmonize_ASDC_2024-08-21.h5ad': 'accae6f5-2277-4237-9283-7aac54e42123',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_Activated_memory_B_cell_2024-08-21.h5ad': '9db395cc-811e-4aa6-a502-6da51700f4d0',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_Adaptive_NK_cell_2024-08-21.h5ad': '16fbf925-061d-48d2-8539-f4063dfe6791',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_BaEoMaP_cell_2024-08-21.h5ad': '17cddbe8-1768-454d-a5c1-47344cd92863',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_C1Qpos_CD16_monocyte_2024-08-21.h5ad': '785e01a1-2fbd-42b3-a560-0dfe91a32f1c',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_CD14pos_cDC2_2024-08-21.h5ad': 'd8c44272-8144-4731-a5b1-21906085e1e3',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_CD27neg_effector_B_cell_2024-08-21.h5ad': '0ed8e78f-c1ea-4689-96e7-544cd89d12dd',
 'mercury-nickel-mendelevium/up1_cluster_harmonize_CD27pos_effector_B_cell_2024-08-21.h5ad': 'c0cf6566-01f4-4a2e-8b7c-bb

In [25]:
len(h5ad_uuids)

70

## Download and sort .h5ad files

Sorting the .h5ad files by AIFI_L2 will make reading each cell type much faster by placing cells of the same type next to each other in the sparse matrix.

In [26]:

# run once
for uuid in h5ad_uuids.values():
    sort_adata_uuid(uuid, sort_cols = ['AIFI_L3'])

downloading fileID: accae6f5-2277-4237-9283-7aac54e42123
Files have been successfully downloaded!
downloading fileID: 9db395cc-811e-4aa6-a502-6da51700f4d0
Files have been successfully downloaded!
downloading fileID: 16fbf925-061d-48d2-8539-f4063dfe6791
Files have been successfully downloaded!
downloading fileID: 17cddbe8-1768-454d-a5c1-47344cd92863
Files have been successfully downloaded!
downloading fileID: 785e01a1-2fbd-42b3-a560-0dfe91a32f1c
Files have been successfully downloaded!
downloading fileID: d8c44272-8144-4731-a5b1-21906085e1e3
Files have been successfully downloaded!
downloading fileID: 0ed8e78f-c1ea-4689-96e7-544cd89d12dd
Files have been successfully downloaded!
downloading fileID: c0cf6566-01f4-4a2e-8b7c-bb80c061b2ba
Files have been successfully downloaded!
downloading fileID: dd45daa4-c391-4b5f-9b81-82dcf8dba9c1
Files have been successfully downloaded!
downloading fileID: 0da3ffa6-5a87-4861-a547-8daae37b5fd9
Files have been successfully downloaded!
downloading fileID: 

## Open connections to .h5ad files

Now that they're sorted, we can open these files with on-disk backing so we don't have to read the entire file at once.

## Process files

In [36]:
input_path = "/home/jupyter/old_cache/**/"

In [37]:
### read in processed leiden adata
filenames = get_filepaths_with_glob(input_path, "up1_cluster_harmonize_*.h5ad")  
filenames[:5]
len(filenames)

70

In [38]:
### extract cell types
# Apply the function to each filename in the list using list comprehension
cell_types = [extract_substring(path,pattern = r'up1_cluster_harmonize_(.*?)_\d{4}-\d{2}-\d{2}\.h5ad') for path in filenames]
cell_types[:5]

['CD8_MAIT',
 'ISGpos_CD14_monocyte',
 'Early_memory_B_cell',
 'CMP_cell',
 'KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell']

In [39]:
file_dict = dict(zip(cell_types, filenames))
list(file_dict.items())[:10]

[('CD8_MAIT',
  '/home/jupyter/old_cache/58bf6faf-7fbb-4a3a-85f0-c561116b7eb9/up1_cluster_harmonize_CD8_MAIT_2024-08-21.h5ad'),
 ('ISGpos_CD14_monocyte',
  '/home/jupyter/old_cache/11c57b90-dc55-4640-a945-6713d191db92/up1_cluster_harmonize_ISGpos_CD14_monocyte_2024-08-22.h5ad'),
 ('Early_memory_B_cell',
  '/home/jupyter/old_cache/8876f435-364f-429f-a1aa-e64238ef00bf/up1_cluster_harmonize_Early_memory_B_cell_2024-08-22.h5ad'),
 ('CMP_cell',
  '/home/jupyter/old_cache/4a4eed4b-2f5a-455d-91d5-26c06c6405d9/up1_cluster_harmonize_CMP_cell_2024-08-21.h5ad'),
 ('KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell',
  '/home/jupyter/old_cache/5d525228-f4d3-4fa3-aebc-098f82dcc8c2/up1_cluster_harmonize_KLRF1neg_GZMBpos_CD27neg_EM_CD8_T_cell_2024-08-22.h5ad'),
 ('ISGpos_naive_CD8_T_cell',
  '/home/jupyter/old_cache/62e01915-7a5c-479d-91b1-e8ad2fbfdd3d/up1_cluster_harmonize_ISGpos_naive_CD8_T_cell_2024-08-23.h5ad'),
 ('pDC',
  '/home/jupyter/old_cache/21647c22-b980-4f8f-ace5-41e49edf5a81/up1_cluster_harmonize_p

## Read in clinical metadata

In [42]:
### read in 144 pre-ra clinical samples
meta_sample_selection = pd.read_csv("/home/jupyter/preprocessing_output/UP1_218scRNAsamp_noDuplicates.csv")
meta_sample_selection

,Unnamed: 0,lastUpdated,sample.id,sample.bridgingControl,sample.sampleKitGuid,sample.visitName,sample.visitDetails,sample.drawDate,sample.daysSinceFirstVisit,sample.diseaseStatesRecordedAtVisit,...,subject.subjectGuid,cohort.cohortGuid,file.userTags.details,file.userTags.group,file.userTags.name,file.userTags.origin,file.userTags.other,file.userTags.version,specimens.specimenType,specimens.specimenGuid
0,1,2024-08-17T13:57:19.462Z,3d19d773-6d5a-473d-b610-9221f1ebd921,False,KT00809,Flu Year 1 Pre-Vac 7-12 Weeks,N/A - Flu-Series Timepoint Only,2020-08-01T00:00:00Z,0,NaN,...,UP1008,UP1,NaN,NaN,NaN,NaN,NaN,NaN,PBMC,PB00809-06
1,2,2024-08-17T13:57:19.462Z,d8de83f3-efd9-4a06-ae84-0722349de493,False,KT00813,Flu Year 1 Day 7,N/A - Flu-Series Timepoint Only,2020-11-01T00:00:00Z,57,NaN,...,UP1012,UP1,NaN,NaN,NaN,NaN,NaN,NaN,PBMC,PB00813-02
2,3,2024-08-17T13:57:19.462Z,cb3fef48-6d71-4394-96a1-04e66a511d51,False,KT00815,Flu Year 1 Day 7,N/A - Flu-Series Timepoint Only,2020-11-01T00:00:00Z,62,NaN,...,UP1007,UP1,NaN,NaN,NaN,NaN,NaN,NaN,Plasma,PL00815-19
3,4,2024-08-17T13:57:19.462Z,ffd153fb-c676-42bf-93c6-ccac3d3e6c75,False,KT00824,Flu Year 1 Day 7,N/A - Flu-Series Timepoint Only,2020-11-01T00:00:00Z,62,NaN,...,UP1013,UP1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PL00824-21
4,5,2024-08-17T13:57:19.462Z,3563c387-a91b-4584-b69d-1ebe3d7db641,False,KT00826,Flu Year 1 Day 90,N/A - Flu-Series Timepoint Only,2020-12-01T00:00:00Z,114,NaN,...,UP1001,UP1,NaN,NaN,NaN,NaN,NaN,NaN,PBMC,PB00826-07
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213,214,2024-08-17T13:57:19.462Z,2f692d87-2bd1-4bab-a83b-1b398c8e82f8,False,KT03008,COVID-19 Visit 3,N/A - Flu-Series Timepoint Only,2021-12-01T00:00:00Z,260,NaN,...,UP1022,UP1,NaN,NaN,NaN,NaN,NaN,NaN,Plasma,PL03008-019
214,215,2024-08-17T13:57:19.462Z,a235e412-10fd-4b0f-906b-2607c376a377,False,KT02917,COVID-19 Visit 6,N/A - Flu-Series Timepoint Only,2021-11-01T00:00:00Z,432,NaN,...,UP1003,UP1,NaN,NaN,NaN,NaN,NaN,NaN,Plasma,PL02917-005
215,216,2024-08-17T13:57:19.462Z,8a54d28b-784d-4169-950e-48f43682584c,False,KT03029,COVID-19 Visit 6,N/A - Flu-Series Timepoint Only,2021-12-01T00:00:00Z,282,NaN,...,UP1022,UP1,NaN,NaN,NaN,NaN,NaN,NaN,Plasma,PL03029-016
216,217,2024-08-17T13:57:19.462Z,a70a645b-7491-4d16-af0f-dc0a3115ab27,False,KT03016,COVID-19 Visit 6,N/A - Flu-Series Timepoint Only,2021-12-01T00:00:00Z,105,NaN,...,UP1031,UP1,NaN,NaN,NaN,NaN,NaN,NaN,Plasma,PL03016-014


In [44]:
## check DTC is under -750
meta_sample_selection[['days_to_conversion']].describe()

In [45]:
meta_sample_selection.columns

Index(['Unnamed: 0', 'lastUpdated', 'sample.id', 'sample.bridgingControl',
       'sample.sampleKitGuid', 'sample.visitName', 'sample.visitDetails',
       'sample.drawDate', 'sample.daysSinceFirstVisit',
       'sample.diseaseStatesRecordedAtVisit', 'file.id', 'file.name',
       'file.batchID', 'file.panel', 'file.pool', 'file.fileType',
       'file.majorVersion', 'subject.id', 'subject.biologicalSex',
       'subject.birthYear', 'subject.ethnicity', 'subject.partnerCode',
       'subject.race', 'subject.subjectGuid', 'cohort.cohortGuid',
       'file.userTags.details', 'file.userTags.group', 'file.userTags.name',
       'file.userTags.origin', 'file.userTags.other', 'file.userTags.version',
       'specimens.specimenType', 'specimens.specimenGuid'],
      dtype='object')

In [48]:
meta_sample_selection = meta_sample_selection[[ 'sample.sampleKitGuid', 'subject.subjectGuid', 'sample.visitName','subject.id', 'subject.biologicalSex',
       'subject.birthYear', 'subject.ethnicity', 'subject.race']]

meta_sample_selection

,sample.sampleKitGuid,subject.subjectGuid,sample.visitName,subject.id,subject.biologicalSex,subject.birthYear,subject.ethnicity,subject.race
0,KT00809,UP1008,Flu Year 1 Pre-Vac 7-12 Weeks,06a0c838-f555-41a6-af73-075dfe44463a,Female,2009,Non-Hispanic origin,Caucasian
1,KT00813,UP1012,Flu Year 1 Day 7,ce64a455-05e0-470d-abbd-b7e8cc899923,Female,2009,Non-Hispanic origin,Caucasian
2,KT00815,UP1007,Flu Year 1 Day 7,741ad2b7-c974-4dce-aa7a-a58da094c32f,Female,2009,Non-Hispanic origin,Caucasian
3,KT00824,UP1013,Flu Year 1 Day 7,012944ea-c521-40f5-9922-d311835ee9fd,Male,2009,Non-Hispanic origin,African American
4,KT00826,UP1001,Flu Year 1 Day 90,48e2432b-a0da-4deb-9954-4d7c3434a82a,Female,2009,Non-Hispanic origin,African American
...,...,...,...,...,...,...,...,...
213,KT03008,UP1022,COVID-19 Visit 3,724c7a6f-9060-403d-a670-7a987635fac4,Male,2010,Non-Hispanic origin,Caucasian
214,KT02917,UP1003,COVID-19 Visit 6,ac54e15a-a532-4069-aebe-a6870e77f3c3,Female,2009,Non-Hispanic origin,Caucasian
215,KT03029,UP1022,COVID-19 Visit 6,724c7a6f-9060-403d-a670-7a987635fac4,Male,2010,Non-Hispanic origin,Caucasian
216,KT03016,UP1031,COVID-19 Visit 6,2021e780-b83e-4a39-b090-ef2e777ccae5,Female,2010,Non-Hispanic origin,Caucasian


## Process Each Cell Types

In [49]:
### create unit test: cell type with no renaming, cell type with 1 rename and cell type with multiple rename
#subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'ASDC', 'CM_CD4_T_cell'))
subfile_dict = dict((k, file_dict[k]) for k in ('CD4_MAIT', 'Core_naive_CD4_T_cell'))

subfile_dict

{'CD4_MAIT': '/home/jupyter/old_cache/dd45daa4-c391-4b5f-9b81-82dcf8dba9c1/up1_cluster_harmonize_CD4_MAIT_2024-08-21.h5ad',
 'Core_naive_CD4_T_cell': '/home/jupyter/old_cache/6eaa8262-cca7-40d7-ab65-f01e8271f265/up1_cluster_harmonize_Core_naive_CD4_T_cell_2024-08-22.h5ad'}

In [50]:
cell_types_list = []
cell_types_list

[]

In [51]:
for label, file in file_dict.items():
    print("Key:", label)
    print("Value:", file)

    adata = sc.read_h5ad(file)

    # restore raw counts to X
    adata = adata.raw.to_adata()
    # delete raw layer
    adata.raw = None

    #print(adata)
    
    meta = adata.obs

    print("Unfiltered metadata shape:" + str(meta.shape))
    
    # Perform the merge
    meta2 = pd.merge(meta, meta_sample_selection, how='inner', on=['sample.sampleKitGuid'])
    
    print("Filtered metadata shape " + str(meta2.shape))
    
    print("Number of unique clinical sample selected: "+ str(len(meta2['sample.sampleKitGuid'].unique())) )
    
    # subset anndata by 
    adata_subset = adata[adata.obs['sample.sampleKitGuid'].isin(meta2['sample.sampleKitGuid'])]

    adata_subset_before = adata_subset
    
    #adata_subset.obs = meta2.set_index(adata_subset.obs.index) -- omit
    # adata_subset.obs = meta2.set_index(['barcodes'])
    print(adata_subset)
    
    # append to outfile list
    cell_types_list.append(adata_subset)

Key: CD8_MAIT
Value: /home/jupyter/old_cache/58bf6faf-7fbb-4a3a-85f0-c561116b7eb9/up1_cluster_harmonize_CD8_MAIT_2024-08-21.h5ad
Unfiltered metadata shape:(69752, 49)
Filtered metadata shape (69752, 56)
Number of unique clinical sample selected: 218
View of AnnData object with n_obs × n_vars = 69752 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_

In [35]:
cell_types_list[:3]

[View of AnnData object with n_obs × n_vars = 9291 × 33538
     obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'sample.diseaseStatesRecordedAtVisit', 'sample.daysSinceFirstVisit', 'file.id', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2', 'doublets_manual', 'AIFI_L3_n

In [52]:
len(cell_types_list)

70

## Concatenate cell types into single object

In [53]:
# Concatenate all aligned AnnData objects in the list
adata_combined = ad.concat(cell_types_list, axis=0)

In [54]:
adata_combined

AnnData object with n_obs × n_vars = 3743041 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2'
    obsm: 'X_pca', 'X_pca_harmony', 'X

In [55]:
adata_combined.obs

,barcodes,batch_id,cell_name,cell_uuid,chip_id,hto_barcode,hto_category,n_genes,n_mito_umis,n_reads,...,total_counts,log1p_total_counts,pct_counts_in_top_50_genes,pct_counts_in_top_100_genes,pct_counts_in_top_200_genes,pct_counts_in_top_500_genes,total_counts_mito,log1p_total_counts_mito,pct_counts_mito,leiden_harmony_2
barcodes,,,,,,,,,,,,,,,,,,,,,
2d9c18d04e0711ec865c16f89d689413,2d9c18d04e0711ec865c16f89d689413,B094,carbonless_stonelike_paca,2d9c18d04e0711ec865c16f89d689413,B094-P2C2,GTCAACTCTTTAGCG,singlet,1664,126,19290,...,5622,8.634621,42.280327,56.652437,66.364283,78.637496,126,4.844187,2.241195,7
b50dac482e1b11ee9ddc7ed4db856882,b50dac482e1b11ee9ddc7ed4db856882,B132,soft_entitled_bats,b50dac482e1b11ee9ddc7ed4db856882,B132-P1C1,AAATCTCTCAGGCTC,singlet,1859,209,16014,...,5429,8.599694,37.023393,49.143489,58.832197,72.389022,209,5.347108,3.849696,15
b50d4ad22e1b11ee9ddc7ed4db856882,b50d4ad22e1b11ee9ddc7ed4db856882,B132,insidious_ununhexium_kinkajou,b50d4ad22e1b11ee9ddc7ed4db856882,B132-P1C1,AAATCTCTCAGGCTC,singlet,1567,165,7847,...,3308,8.104401,29.262394,38.301088,48.669891,67.744861,165,5.111988,4.987908,1
b50d42622e1b11ee9ddc7ed4db856882,b50d42622e1b11ee9ddc7ed4db856882,B132,nimble_parallel_narwhal,b50d42622e1b11ee9ddc7ed4db856882,B132-P1C1,AAATCTCTCAGGCTC,singlet,1186,135,9587,...,3588,8.185629,45.234114,58.472687,68.924192,80.880713,135,4.912655,3.762542,0
b50770bc2e1b11ee9ddc7ed4db856882,b50770bc2e1b11ee9ddc7ed4db856882,B132,bubbly_gangrenous_kawala,b50770bc2e1b11ee9ddc7ed4db856882,B132-P1C1,AAATCTCTCAGGCTC,singlet,1614,187,14860,...,5304,8.576405,40.365762,54.656863,65.365762,78.318250,187,5.236442,3.525641,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
297a73ec608411edb6ad6a209a0d6007,297a73ec608411edb6ad6a209a0d6007,B138,dusty_disyllabic_yaffle,297a73ec608411edb6ad6a209a0d6007,B138-P1C1,TTCCGCCTCTCTTTG,singlet,1165,89,10938,...,4047,8.305978,49.394613,63.998023,72.794663,83.568075,89,4.499810,2.199160,12
43aa15189ab511ed914832f332ae8ac1,43aa15189ab511ed914832f332ae8ac1,B148,chiasmic_rejectable_dachshund,43aa15189ab511ed914832f332ae8ac1,B148-P1C2,AAATCTCTCAGGCTC,singlet,1229,81,14497,...,4554,8.423981,50.856390,65.305226,73.825209,83.992095,81,4.406719,1.778656,5
20012a1e4d4711ecb167fed23aaa2907,20012a1e4d4711ecb167fed23aaa2907,B093,demoded_darkish_grayfox,20012a1e4d4711ecb167fed23aaa2907,B093-P2C3,AAGTATCGTTTCGCA,singlet,1357,46,15399,...,5427,8.599326,52.404643,66.795651,74.405749,84.208587,46,3.850148,0.847614,5


In [ ]:
adata.obs

In [57]:
adata

AnnData object with n_obs × n_vars = 40775 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2'
    uns: 'hvg', 'leiden', 'log1p', 'neig

In [58]:
#### test without randomonization

# get test data
joint_adata_test = adata_combined[adata_combined.obs['barcodes'].isin(adata.obs.index)].copy()
#adata_subset2 = adata_subset[adata_subset.obs['barcodes'].isin(random_barcodes)].copy()
joint_adata_test

AnnData object with n_obs × n_vars = 40775 × 33538
    obs: 'barcodes', 'batch_id', 'cell_name', 'cell_uuid', 'chip_id', 'hto_barcode', 'hto_category', 'n_genes', 'n_mito_umis', 'n_reads', 'n_umis', 'original_barcodes', 'pbmc_sample_id', 'pool_id', 'well_id', 'sample.sampleKitGuid', 'cohort.cohortGuid', 'subject.subjectGuid', 'subject.biologicalSex', 'subject.race', 'subject.ethnicity', 'subject.birthYear', 'sample.visitName', 'sample.drawDate', 'specimens.specimenGuid', 'specimens.specimenType', 'file.id', 'file.batchID', 'subset_grp', 'predicted_doublet', 'doublet_score', 'AIFI_L1', 'AIFI_L1_score', 'AIFI_L2', 'AIFI_L2_score', 'AIFI_L3', 'AIFI_L3_score', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mito', 'log1p_total_counts_mito', 'pct_counts_mito', 'leiden_harmony_2'
    obsm: 'X_pca', 'X_pca_harmony', 'X_u

In [59]:
cert_exprs = joint_adata_test.to_df()
cert_exprs.index=joint_adata_test.obs['barcodes']
cert_exprs = cert_exprs.sort_index()

In [60]:
cd4na_exprs = adata.to_df()
cd4na_exprs.index = adata.obs['barcodes']
cd4na_exprs = cd4na_exprs.sort_index()


In [61]:
cert_exprs.shape

(40775, 33538)

In [62]:
cd4na_exprs.shape

(40775, 33538)

In [63]:
(cd4na_exprs == cert_exprs).all(axis=1).value_counts()

True    40775
Name: count, dtype: int64

In [64]:
(cd4na_exprs == cert_exprs).all(axis=0).value_counts()


True    33538
Name: count, dtype: int64

In [65]:
### test with randomization
# find share barcodes in all 3 datase
share_barcodes = set(adata_combined.obs['barcodes']) & set(adata.obs['barcodes'])
share_genes = set(adata_combined.var_names) & set(adata.var_names) 

# select 10000 cells randomly
random_barcodes = random.sample(sorted(share_barcodes), k=10000)

In [66]:
# get test data
joint_adata_test = adata_combined[adata_combined.obs['barcodes'].isin(random_barcodes), 
    adata_combined.var_names.isin(share_genes)].to_memory()
cd4na_adata_test = adata[adata.obs['barcodes'].isin(random_barcodes), 
    adata.var_names.isin(share_genes)].to_memory()
    

In [70]:
joint_adata_test.raw

In [73]:
# move raw counts to X
#joint_adata_test.raw = joint_adata_test
#cd4na_adata_test.raw = cd4na_adata_test

joint_adata_test = joint_adata_test.raw.to_adata()
cd4na_adata_test= cd4na_adata_test.raw.to_adata()

In [74]:
cert_exprs = joint_adata_test.to_df()
cert_exprs.index=joint_adata_test.obs['barcodes']
cert_exprs = cert_exprs.sort_index()

In [75]:
cd4na_exprs = cd4na_adata_test.to_df()
cd4na_exprs.index = cd4na_adata_test.obs['barcodes']
cd4na_exprs = cd4na_exprs.sort_index()

In [76]:
(cd4na_exprs.index == cert_exprs.index).all()


True

In [77]:
(cd4na_exprs.columns == cert_exprs.columns).all()

True

In [78]:
(cd4na_exprs == cert_exprs).all(axis=0).value_counts()


True    33538
Name: count, dtype: int64

In [79]:
(cd4na_exprs == cert_exprs).all(axis=1).value_counts()

True    10000
Name: count, dtype: int64

In [80]:
out_file = 'output/up1_dc_sample_selection_combined_adata_{d}.h5ad'.format(
        d = date.today()
    )
out_file

'output/up1_dc_sample_selection_combined_adata_2024-08-29.h5ad'

In [81]:
### save
adata_combined.write_h5ad(out_file)

## Upload Cell Type data to HISE

Finally, we'll use `hisepy.upload.upload_files()` to send a copy of our output to HISE to use for downstream analysis steps.

In [96]:
ss = hisepy.get_study_spaces()
study_space_uuid = ss[0]['id']
title = '07_2 PBMC L3 Sample Selection Combined Anndata {d}'.format(d = date.today())
title

'07_2 PBMC L3 Sample Selection Combined Anndata 2024-08-29'

In [91]:
study_space_uuid

'0b6bf907-6985-40e0-944d-677ac932677f'

In [92]:
search_id = element_id()
search_id

'moscovium-iron-fermium'

In [93]:
in_files = list(h5ad_uuids.values())
in_files

['accae6f5-2277-4237-9283-7aac54e42123',
 '9db395cc-811e-4aa6-a502-6da51700f4d0',
 '16fbf925-061d-48d2-8539-f4063dfe6791',
 '17cddbe8-1768-454d-a5c1-47344cd92863',
 '785e01a1-2fbd-42b3-a560-0dfe91a32f1c',
 'd8c44272-8144-4731-a5b1-21906085e1e3',
 '0ed8e78f-c1ea-4689-96e7-544cd89d12dd',
 'c0cf6566-01f4-4a2e-8b7c-bb80c061b2ba',
 'dd45daa4-c391-4b5f-9b81-82dcf8dba9c1',
 '0da3ffa6-5a87-4861-a547-8daae37b5fd9',
 '58bf6faf-7fbb-4a3a-85f0-c561116b7eb9',
 'b35b263c-3e0a-419c-8f86-4ee64eebbe36',
 '388cb495-df9e-449f-a406-fed165445a59',
 '1403e5b5-35d4-49ae-b122-83d32a27f4a2',
 '4a4eed4b-2f5a-455d-91d5-26c06c6405d9',
 '0cff6a24-3f00-4ad8-853b-3d3163589f8f',
 'd90b222b-4c24-4cd3-a74a-821959418e00',
 '8251dc12-47f1-4d4d-bdaa-d7a2b9ceb242',
 '0714163d-dcb7-4695-97c7-b85d8daad111',
 '46a2ee7d-5b73-4139-afd4-fdad3fa48154',
 '4dfb0616-0b14-4218-885b-5122d9102134',
 '6eaa8262-cca7-40d7-ab65-f01e8271f265',
 'cac7f6e7-ec0b-4b1c-8e0f-21f6b5426c63',
 'c46415db-e19c-471b-ac81-28a2fc6aa4e7',
 '8876f435-364f-

In [94]:
out_file

'output/up1_dc_sample_selection_combined_adata_2024-08-29.h5ad'

In [97]:
hisepy.upload.upload_files(
    files = [out_file],
    study_space_id = study_space_uuid,
    title = title,
    input_file_ids = in_files,
    destination = search_id
)

you are trying to upload file_ids... ['output/up1_dc_sample_selection_combined_adata_2024-08-29.h5ad']. Do you truly want to proceed?


(y/n) y


{'trace_id': 'ea1a250f-1684-440e-be12-ef176185ba04',
 'files': ['output/up1_dc_sample_selection_combined_adata_2024-08-29.h5ad']}

In [95]:
import session_info
session_info.show()